# Bài 2
Đây là notebook chứa mã nguồn đầy đủ của bài 2.

In [ ]:
from ipynb.fs.full.notebooks.bai01_notebook import _solve


In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import pulp
import pyomo.environ as pyo
from scipy.optimize import linprog, minimize, milp, LinearConstraint, Bounds
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize as pymoo_minimize

from src.data_loader import get_data


In [ ]:
def solve_bai02(budget=100, min_I=25, min_AI=15, coef_I=0.85, coef_AI=1.20, coef_H=0.95, coef_RD=1.35):
    labels = ['I Hạ tầng', 'AI Dữ liệu', 'H Nhân lực', 'R&D']

    # === (1) Giải bằng scipy.optimize.linprog ===
    x, z, ok = _solve(budget, min_I, min_AI, coef_I, coef_AI, coef_H, coef_RD)

    # === (2) Giải bằng PuLP + Dual values ===
    m = pulp.LpProblem('Budget_LP_Dual', pulp.LpMaximize)
    xv = [pulp.LpVariable(f'x{i}', lowBound=0) for i in range(4)]
    coefs = [coef_I, coef_AI, coef_H, coef_RD]
    m += pulp.lpSum(coefs[i] * xv[i] for i in range(4))

    # Constraints (named so we can extract duals)
    c1 = m.addConstraint(pulp.lpSum(xv) <= budget, name='C1_Budget')
    c2 = m.addConstraint(0.35*xv[0] - 0.65*xv[1] + 0.35*xv[2] - 0.65*xv[3] <= 0, name='C2_Tech35')
    c3 = m.addConstraint(xv[0] >= min_I, name='C3_MinI')
    c4 = m.addConstraint(xv[1] >= min_AI, name='C4_MinAI')
    c5 = m.addConstraint(xv[2] >= 20, name='C5_MinH')
    c6 = m.addConstraint(xv[3] >= 10, name='C6_MinRD')

    m.solve(pulp.PULP_CBC_CMD(msg=False))
    pulp_ok = m.status == pulp.LpStatusOptimal

    dual_values = {}
    pulp_alloc = {}
    pulp_z = 0.0
    if pulp_ok:
        pulp_z = pulp.value(m.objective)
        pulp_alloc = {labels[i]: round(pulp.value(xv[i]), 2) for i in range(4)}
        for name, constraint in m.constraints.items():
            dual_values[name] = round(constraint.pi if constraint.pi is not None else 0.0, 4)

    # === (3) Phân tích độ nhạy: Z*(B) ===
    budget_range = sorted(set([80, 100, 120, 140, budget]))
    sensitivity_budgets = []
    sensitivity_z = []
    for bb in budget_range:
        _, zz, _ = _solve(bb, min_I, min_AI, coef_I, coef_AI, coef_H, coef_RD)
        sensitivity_budgets.append(bb)
        sensitivity_z.append(zz)

    # === (4) Kịch bản x₃ ≥ 30 ===
    c_scenario = [-coef_I, -coef_AI, -coef_H, -coef_RD]
    A_ub_s = [[1, 1, 1, 1], [0.35, -0.65, 0.35, -0.65]]
    b_ub_s = [budget, 0]
    bounds_s = [(min_I, None), (min_AI, None), (30, None), (10, None)]  # x₃ ≥ 30
    res_s = linprog(c_scenario, A_ub=A_ub_s, b_ub=b_ub_s, bounds=bounds_s, method='highs')
    if res_s.success:
        scenario_x3 = {'status': 'Optimal', 'Z': round(float(-res_s.fun), 2),
                        'allocation': dict(zip(labels, [round(v, 2) for v in res_s.x.tolist()]))}
    else:
        scenario_x3 = {'status': 'Infeasible', 'Z': 0.0, 'allocation': dict(zip(labels, [0]*4))}

    return {
        'status':      'Optimal' if ok else 'Infeasible',
        'Z':           z,
        'allocation':  dict(zip(labels, x)),
        'pulp_z':      pulp_z,
        'pulp_alloc':  pulp_alloc,
        'dual_values': dual_values,
        'sensitivity_budgets': sensitivity_budgets,
        'sensitivity_z':       sensitivity_z,
        'scenario_x3': scenario_x3,
        'feasible':    ok,
    }

In [ ]:
if __name__ == '__main__':
    res = solve_bai02()
    # In ra một số key để kiểm tra
    if isinstance(res, dict):
        print(res.keys())